In [ ]:
from word2number import w2n
from thefuzz import process
import pandas as pd
import numpy as np

In [ ]:
# Cargamos el dataset
df = pd.read_csv('ML_cars.csv')
df.head()

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


In [19]:
# Vemos qué tipo de datos hay en cada columna y si hay valores nulos (NaN)
print("--- INFO DEL DATASET ---")
df.info()

print("\n--- TAMAÑO ---")
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

--- INFO DEL DATASET ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 26 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   car_ID            205 non-null    int64  
 1   symboling         205 non-null    int64  
 2   CarName           205 non-null    object 
 3   fueltype          205 non-null    object 
 4   aspiration        205 non-null    object 
 5   doornumber        205 non-null    object 
 6   carbody           205 non-null    object 
 7   drivewheel        205 non-null    object 
 8   enginelocation    205 non-null    object 
 9   wheelbase         205 non-null    float64
 10  carlength         205 non-null    float64
 11  carwidth          205 non-null    float64
 12  carheight         205 non-null    float64
 13  curbweight        205 non-null    int64  
 14  enginetype        205 non-null    object 
 15  cylindernumber    205 non-null    object 
 16  enginesize        2

In [ ]:
print("=== AUDITORÍA AUTOMÁTICA DEL DATASET ===\n")

# 1. Escáner de Nulos (Porcentaje real)
nulos = df.isnull().mean() * 100
columnas_con_nulos = nulos[nulos > 0]
if not columnas_con_nulos.empty:
    print("⚠️ COLUMNAS CON DATOS FALTANTES (%):")
    print(columnas_con_nulos.sort_values(ascending=False))
else:
    print("✅ No hay datos nulos detectados.")

print("\n-----------------------------------------")

# 2. Escáner de Variables Categóricas
columnas_texto = df.select_dtypes(include=['object']).columns

print(f"🔍 ANALIZANDO {len(columnas_texto)} COLUMNAS DE TEXTO:\n")

for col in columnas_texto:
    valores_unicos = df[col].nunique()
    print(f"Columna '{col}': {valores_unicos} valores distintos.")
    
    # Si la columna tiene menos de 15 valores distintos, imprimimos cuáles son.
    # Esto hace saltar a la vista instantáneamente los errores como "maxda" y "mazda" o "two" y 2.
    if valores_unicos < 15:
        print(f"   ↳ Valores: {df[col].unique()}")
    else:
        print(f"   ↳ (Demasiados valores para imprimir. Podría ser un ID, un nombre libre o requerir limpieza profunda)")
    print("")

print("-----------------------------------------")

# 3. Escáner de Variables Numéricas (Anomalías matemáticas)
# Usamos describe() pero solo miramos los mínimos y máximos buscando absurdos
print("📊 ANOMALÍAS EN NÚMEROS (Min y Max):")
numericas = df.select_dtypes(include=['int64', 'float64']).describe().loc[['min', 'max']]
print(numericas)

=== AUDITORÍA AUTOMÁTICA DEL DATASET ===

✅ No hay datos nulos detectados.

-----------------------------------------
🔍 ANALIZANDO 10 COLUMNAS DE TEXTO:

Columna 'CarName': 147 valores distintos.
   ↳ (Demasiados valores para imprimir. Podría ser un ID, un nombre libre o requerir limpieza profunda)

Columna 'fueltype': 2 valores distintos.
   ↳ Valores: ['gas' 'diesel']

Columna 'aspiration': 2 valores distintos.
   ↳ Valores: ['std' 'turbo']

Columna 'doornumber': 2 valores distintos.
   ↳ Valores: ['two' 'four']

Columna 'carbody': 5 valores distintos.
   ↳ Valores: ['convertible' 'hatchback' 'sedan' 'wagon' 'hardtop']

Columna 'drivewheel': 3 valores distintos.
   ↳ Valores: ['rwd' 'fwd' '4wd']

Columna 'enginelocation': 2 valores distintos.
   ↳ Valores: ['front' 'rear']

Columna 'enginetype': 7 valores distintos.
   ↳ Valores: ['dohc' 'ohcv' 'ohc' 'l' 'rotor' 'ohcf' 'dohcv']

Columna 'cylindernumber': 7 valores distintos.
   ↳ Valores: ['four' 'six' 'five' 'three' 'twelve' 'two' '

In [21]:
# 1. Hacemos un experimento temporal: separamos la primera palabra 
# asumiendo que esa es la marca, y la pasamos a minúsculas.
marcas_temporales = df['CarName'].apply(lambda x: str(x).split(' ')[0].lower())

# 2. Le pedimos a Pandas que nos muestre todos los valores únicos de esa primera palabra
print("🔍 MARCAS DETECTADAS EN LA INVESTIGACIÓN FOCALIZADA:")
print(marcas_temporales.unique())

# 3. (Opcional pero letal) Contamos cuántas veces aparece cada una
print("\n📊 CONTEO DE CADA MARCA (Para cazar errores raros):")
print(marcas_temporales.value_counts())

🔍 MARCAS DETECTADAS EN LA INVESTIGACIÓN FOCALIZADA:
['alfa-romero' 'audi' 'bmw' 'chevrolet' 'dodge' 'honda' 'isuzu' 'jaguar'
 'maxda' 'mazda' 'buick' 'mercury' 'mitsubishi' 'nissan' 'peugeot'
 'plymouth' 'porsche' 'porcshce' 'renault' 'saab' 'subaru' 'toyota'
 'toyouta' 'vokswagen' 'volkswagen' 'vw' 'volvo']

📊 CONTEO DE CADA MARCA (Para cazar errores raros):
CarName
toyota         31
nissan         18
mazda          15
honda          13
mitsubishi     13
subaru         12
peugeot        11
volvo          11
dodge           9
volkswagen      9
bmw             8
buick           8
audi            7
plymouth        7
saab            6
isuzu           4
porsche         4
alfa-romero     3
jaguar          3
chevrolet       3
vw              2
maxda           2
renault         2
toyouta         1
vokswagen       1
mercury         1
porcshce        1
Name: count, dtype: int64


In [22]:
print("--- INICIANDO CORRECCIÓN AUTOMÁTICA CON NLP ---")

# 1. Creamos la columna 'Marca' recortando la primera palabra de 'CarName'
df['Marca'] = df['CarName'].apply(lambda x: str(x).split(' ')[0].lower())

# 2. "Base de Datos Maestra" (Las marcas correctas)
marcas_conocidas = [
    'alfa-romeo', 'audi', 'bmw', 'buick', 'chevrolet', 'dodge', 'honda', 
    'isuzu', 'jaguar', 'mazda', 'mercury', 'mitsubishi', 'nissan', 
    'peugeot', 'plymouth', 'porsche', 'renault', 'saab', 'subaru', 
    'toyota', 'volkswagen', 'volvo'
]

# 3. Nuestro "Motor de Corrección"
def corregir_ortografia(palabra_sucia, lista_maestra, umbral_minimo=80):
    mejor_coincidencia, puntaje = process.extractOne(palabra_sucia, lista_maestra)
    
    if puntaje >= umbral_minimo:
        return mejor_coincidencia
    else:
        return palabra_sucia

# 4. Aplicamos el motor a la columna 'Marca' que acabamos de crear
df['Marca_Limpia'] = df['Marca'].apply(lambda x: corregir_ortografia(x, marcas_conocidas))

# 5. Verificamos la magia y limpiamos la basura
print("Antes de thefuzz:", df['Marca'].unique()[:20]) 
print("Después de thefuzz:", df['Marca_Limpia'].unique()[:20])

--- INICIANDO CORRECCIÓN AUTOMÁTICA CON NLP ---
Antes de thefuzz: ['alfa-romero' 'audi' 'bmw' 'chevrolet' 'dodge' 'honda' 'isuzu' 'jaguar'
 'maxda' 'mazda' 'buick' 'mercury' 'mitsubishi' 'nissan' 'peugeot'
 'plymouth' 'porsche' 'porcshce' 'renault' 'saab']
Después de thefuzz: ['alfa-romeo' 'audi' 'bmw' 'chevrolet' 'dodge' 'honda' 'isuzu' 'jaguar'
 'mazda' 'buick' 'mercury' 'mitsubishi' 'nissan' 'peugeot' 'plymouth'
 'porsche' 'renault' 'saab' 'subaru' 'toyota']


In [ ]:
print("--- INICIANDO PROTOCOLO DE LIMPIEZA FINAL ---")

# 1. Eliminamos ruido (El ID no sirve para predecir)
df = df.drop('car_ID', axis=1, errors='ignore')

# 2. Traducción Dinámica de Textos a Números Enteros
df['doornumber'] = df['doornumber'].apply(w2n.word_to_num)
df['cylindernumber'] = df['cylindernumber'].apply(w2n.word_to_num)
print("✅ Textos convertidos dinámicamente a variables numéricas (int64).")

# 3. CREACIÓN DE LA VARIABLE OBJETIVO (Para el modelo de Clasificación)
mediana_precio = df['price'].median()

# Creamos una nueva columna: 1 si es más caro que la mediana, 0 si es más barato
df['categoria_precio'] = (df['price'] > mediana_precio).astype(int)
print(f"✅ Variable objetivo creada. Mediana calculada en: ${mediana_precio}")

print("\n--- MUESTRA DEL DATASET LIMPIO ---")
print(df[['Marca_Limpia', 'price', 'categoria_precio']].head(10))

--- INICIANDO PROTOCOLO DE LIMPIEZA FINAL ---
✅ Textos convertidos dinámicamente a variables numéricas (int64).
✅ Variable objetivo creada. Mediana calculada en: $10295.0

--- MUESTRA DEL DATASET LIMPIO ---
  Marca_Limpia      price  categoria_precio
0   alfa-romeo  13495.000                 1
1   alfa-romeo  16500.000                 1
2   alfa-romeo  16500.000                 1
3         audi  13950.000                 1
4         audi  17450.000                 1
5         audi  15250.000                 1
6         audi  17710.000                 1
7         audi  18920.000                 1
8         audi  23875.000                 1
9         audi  17859.167                 1


In [ ]:
print("--- FASE FINAL DE PREPARACIÓN DE DATOS (ENCODING) ---")

# 1. Limpieza final de columnas redundantes
df = df.drop(['CarName', 'Marca'], axis=1)

# Renombramos 'Marca_Limpia' a algo estándar
df = df.rename(columns={'Marca_Limpia': 'marca'})

# 2. Identificamos qué columnas siguen siendo de texto (object)
columnas_categoricas = df.select_dtypes(include=['object']).columns
print(f"Transformando las siguientes columnas a binario: {list(columnas_categoricas)}\n")

# 3. Aplicamos One-Hot Encoding
# Si un auto no es 'gas', el algoritmo deduce por descarte que es 'diesel'. No hace falta una columna extra para 'diesel'.
df_final = pd.get_dummies(df, columns=columnas_categoricas, drop_first=True)

# 4. Pandas a veces devuelve True/False. Los pasamos a 1 y 0 para que los modelos no fallen.
columnas_bool = df_final.select_dtypes(include=['bool']).columns
df_final[columnas_bool] = df_final[columnas_bool].astype(int)

# 5. Verificamos el resultado
print("✅ Transformación completa.")
print(f"Tamaño original del dataset: {df.shape}")
print(f"NUEVO tamaño del dataset: {df_final.shape}\n")

# Vemos cómo quedaron las columnas
print("--- MUESTRA DEL DATASET FINALIZADO ---")
print(df_final.head(3))

--- FASE FINAL DE PREPARACIÓN DE DATOS (ENCODING) ---
Transformando las siguientes columnas a binario: ['fueltype', 'aspiration', 'carbody', 'drivewheel', 'enginelocation', 'enginetype', 'fuelsystem', 'marca']

✅ Transformación completa.
Tamaño original del dataset: (205, 26)
NUEVO tamaño del dataset: (205, 62)

--- MUESTRA DEL DATASET FINALIZADO ---
   symboling  doornumber  wheelbase  carlength  carwidth  carheight  \
0          3           2       88.6      168.8      64.1       48.8   
1          3           2       88.6      168.8      64.1       48.8   
2          1           2       94.5      171.2      65.5       52.4   

   curbweight  cylindernumber  enginesize  boreratio  ...  marca_peugeot  \
0        2548               4         130       3.47  ...              0   
1        2548               4         130       3.47  ...              0   
2        2823               6         152       2.68  ...              0   

   marca_plymouth  marca_porsche  marca_renault  marca_sa

In [26]:
print("--- APLICANDO PARCHE DE QA Y GUARDANDO DATASET ---")

# 1. Fusionamos 'vw' con 'volkswagen'
# Usamos el operador lógico OR (|) para que, si el auto era 'vw', ahora marque 1 en 'volkswagen'
df_final['marca_volkswagen'] = (df_final['marca_volkswagen'] | df_final['marca_vw']).astype(int)

# 2. Eliminamos la columna intrusa
df_final = df_final.drop('marca_vw', axis=1)

print("✅ Abreviatura 'vw' fusionada exitosamente. Nueva cantidad de columnas:", df_final.shape[1])

--- APLICANDO PARCHE DE QA Y GUARDANDO DATASET ---
✅ Abreviatura 'vw' fusionada exitosamente. Nueva cantidad de columnas: 61


In [28]:
# Vemos cómo quedaron las columnas
print("--- MUESTRA DEL DATASET FINALIZADO ---")
print(df_final.tail(3))

--- MUESTRA DEL DATASET FINALIZADO ---
     symboling  doornumber  wheelbase  carlength  carwidth  carheight  \
202         -1           4      109.1      188.8      68.9       55.5   
203         -1           4      109.1      188.8      68.9       55.5   
204         -1           4      109.1      188.8      68.9       55.5   

     curbweight  cylindernumber  enginesize  boreratio  ...  marca_nissan  \
202        3012               6         173       3.58  ...             0   
203        3217               6         145       3.01  ...             0   
204        3062               4         141       3.78  ...             0   

     marca_peugeot  marca_plymouth  marca_porsche  marca_renault  marca_saab  \
202              0               0              0              0           0   
203              0               0              0              0           0   
204              0               0              0              0           0   

     marca_subaru  marca_toyota  marca

In [ ]:
# Exportamos la matriz final a un nuevo archivo CSV
nombre_archivo = 'dataset_preparado.csv'
df_final.to_csv(nombre_archivo, index=False)